# Low-Entropy LLM Watermarking

**Baseline**: Soft red-list watermark from Kirchenbauer et al. (ICML 2023)  
Paper: https://arxiv.org/abs/2301.10226  
Original code: https://github.com/jwkirchenbauer/lm-watermarking

This notebook is the interactive entry-point. The core logic lives in:
- `watermark.py` — `WatermarkLogitsProcessor` + `WatermarkDetector`
- `generate.py` — bulk generation on C4 RealNewsLike
- `evaluate.py` — z-scores, PPL, ROC/AUC

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from watermark import WatermarkLogitsProcessor, WatermarkDetector

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "facebook/opt-1.3b"
GAMMA = 0.25
DELTA = 2.0
HASH_KEY = 15485863

print(f"Using device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
).to(DEVICE)
model.eval()
print("Model loaded.")

In [ ]:
vocab_size = len(tokenizer)

watermark_processor = WatermarkLogitsProcessor(
    vocab_size=vocab_size,
    gamma=GAMMA,
    delta=DELTA,
    hash_key=HASH_KEY,
)

detector = WatermarkDetector(
    vocab_size=vocab_size,
    gamma=GAMMA,
    hash_key=HASH_KEY,
    z_threshold=4.0,
)

## Quick demo: single prompt

In [ ]:
PROMPT = (
    "The quick brown fox jumps over the lazy dog. "
    "In recent news, scientists have discovered"
)

@torch.inference_mode()
def generate(prompt: str, watermarked: bool, max_new_tokens: int = 200) -> str:
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(DEVICE)
    processors = [watermark_processor] if watermarked else []
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        logits_processor=processors,
    )
    new_tokens = out[0, input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), new_tokens.tolist()

nw_text, nw_tokens = generate(PROMPT, watermarked=False)
w_text,  w_tokens  = generate(PROMPT, watermarked=True)

print("=== No watermark ===")
print(nw_text[:300])
print()
print("=== Watermarked ===")
print(w_text[:300])

In [ ]:
nw_result = detector.detect(nw_tokens)
w_result  = detector.detect(w_tokens)

print("No-watermark detection:", nw_result)
print("Watermarked detection: ", w_result)

## Bulk generation on C4

Run from terminal (slow — downloads model + dataset):
```bash
python generate.py --num_samples 500 --output_file data/generations.jsonl
```

Then evaluate:
```bash
python evaluate.py --input_file data/generations.jsonl
# add --compute_ppl to also measure oracle perplexity (requires OPT-2.7B)
```

---
## Extensions: Adaptive δ and Forced Generation

Two extensions beyond the Kirchenbauer et al. baseline:
1. **Adaptive δ** — scales the green-list bias with the Shannon entropy of the current distribution. Low-entropy (near-deterministic) steps get δ ≈ 0; high-entropy steps get the full δ.
2. **Forced generation** — continues generating token-by-token until the running z-score exceeds the detection threshold, guaranteeing detectability at the cost of variable output length.


In [ ]:
# Adaptive processor and detector (fixed processor already set up above)
adaptive_processor = WatermarkLogitsProcessor(
    vocab_size=vocab_size,
    gamma=GAMMA,
    delta=DELTA,
    adaptive=True,
    hash_key=HASH_KEY,
)
detector = WatermarkDetector(
    vocab_size=vocab_size,
    gamma=GAMMA,
    hash_key=HASH_KEY,
    z_threshold=4.0,
)
print('Processors and detector ready.')


### Per-token entropy: fixed δ vs adaptive δ

The cell below wraps the logits processor to log the normalized entropy and applied δ at each generation step, then plots them side by side. The key claim: in low-entropy positions the adaptive version applies near-zero bias, while the fixed version always applies the full δ regardless of how predictable the next token is.

In [ ]:
import math, numpy as np, matplotlib.pyplot as plt

class EntropyLoggingProcessor(WatermarkLogitsProcessor):
    """Thin wrapper that records entropy and applied delta at each step."""
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.entropy_log = []
        self.delta_log   = []

    def __call__(self, input_ids, scores):
        norm_h = self._normalized_entropy(scores[0])
        applied = self._adaptive_delta(scores[0]) if self.adaptive else self.delta
        self.entropy_log.append(float(norm_h))
        self.delta_log.append(float(applied))
        return super().__call__(input_ids, scores)

N_DEMO = 100  # tokens to generate for the entropy demo
LOW_ENTROPY_PROMPT = (
    "The capital of France is Paris. The capital of Germany is Berlin. "
    "The capital of Italy is Rome. The capital of Spain is"
)

fixed_logger    = EntropyLoggingProcessor(vocab_size=vocab_size, gamma=GAMMA, delta=DELTA, adaptive=False, hash_key=HASH_KEY)
adaptive_logger = EntropyLoggingProcessor(vocab_size=vocab_size, gamma=GAMMA, delta=DELTA, adaptive=True,  hash_key=HASH_KEY)

@torch.inference_mode()
def generate_logged(prompt, proc, n=N_DEMO):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(ids, max_new_tokens=n, do_sample=True, logits_processor=[proc])
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)

fixed_text    = generate_logged(LOW_ENTROPY_PROMPT, fixed_logger)
adaptive_text = generate_logged(LOW_ENTROPY_PROMPT, adaptive_logger)
print('Fixed:   ', fixed_text[:120])
print('Adaptive:', adaptive_text[:120])


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
steps = range(len(fixed_logger.entropy_log))

axes[0].plot(steps, fixed_logger.entropy_log,    label='Fixed δ',    color='steelblue')
axes[0].plot(steps, adaptive_logger.entropy_log, label='Adaptive δ', color='darkorange', linestyle='--')
axes[0].set_ylabel('Normalized\nEntropy')
axes[0].axhline(0.5, color='gray', linewidth=0.8, linestyle=':')
axes[0].legend()
axes[0].set_title('Per-token normalized entropy (fixed δ vs adaptive δ)')

axes[1].plot(steps, fixed_logger.delta_log,    color='steelblue',  label='Fixed δ applied')
axes[1].plot(steps, adaptive_logger.delta_log, color='darkorange', linestyle='--', label='Adaptive δ applied')
axes[1].set_ylabel('δ applied')
axes[1].legend()

axes[2].fill_between(steps, adaptive_logger.delta_log, DELTA, alpha=0.3, color='green', label='δ saved (low-entropy positions)')
axes[2].set_ylabel('δ reduction')
axes[2].set_xlabel('Token step')
axes[2].legend()

plt.tight_layout()
plt.savefig('entropy_vs_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Avg entropy — Fixed: {np.mean(fixed_logger.entropy_log):.3f}  Adaptive: {np.mean(adaptive_logger.entropy_log):.3f}')
print(f'Avg δ applied — Fixed: {np.mean(fixed_logger.delta_log):.3f}  Adaptive: {np.mean(adaptive_logger.delta_log):.3f}')


### Forced generation: z-score trajectory

Forced generation runs a token-by-token loop, checking the running z-score after each token. The plot below shows how the z-score grows and where it crosses the detection threshold.

In [ ]:
from generate import generate_until_detected

# Capture z-score at each step by re-scoring prefixes
forced_ids = generate_until_detected(
    model, tokenizer.encode(PROMPT, return_tensors='pt')[0].tolist(),
    watermark_processor, detector,
    max_tokens=800, device=DEVICE,
)

z_trajectory = [
    detector.detect(forced_ids[:t+1])['z_score']
    for t in range(1, len(forced_ids))
]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(z_trajectory, color='steelblue', label='Running z-score')
ax.axhline(4.0, color='red', linestyle='--', linewidth=1.2, label='Detection threshold (z=4.0)')
ax.axhline(0,   color='gray', linewidth=0.5)
ax.fill_between(range(len(z_trajectory)), z_trajectory, 4.0,
                where=[z > 4.0 for z in z_trajectory], alpha=0.2, color='green', label='Detected')
ax.set_xlabel('Tokens generated')
ax.set_ylabel('Z-score')
ax.set_title(f'Forced generation z-score trajectory  (stopped at {len(forced_ids)} tokens)')
ax.legend()
plt.tight_layout()
plt.savefig('forced_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Forced length: {len(forced_ids)} tokens  |  final z: {z_trajectory[-1]:.2f}')


### Low-entropy scenario: detection comparison

Run all four conditions on a structured, low-entropy prompt and compare z-scores and detection. Low-entropy prompts (lists, fill-in-the-blank) are where adaptive δ and forced generation are most relevant.

In [ ]:
STRUCTURED_PROMPT = (
    "Country: France  Capital: Paris\n"
    "Country: Germany Capital: Berlin\n"
    "Country: Japan   Capital: Tokyo\n"
    "Country: Brazil  Capital:"
)
N_STRUCT = 80

@torch.inference_mode()
def gen(prompt, proc=None, n=N_STRUCT):
    ids = tokenizer.encode(prompt, return_tensors='pt').to(DEVICE)
    out = model.generate(ids, max_new_tokens=n, do_sample=True,
                         logits_processor=[proc] if proc else [])
    return out[0, ids.shape[1]:].tolist()

prompt_ids_struct = tokenizer.encode(STRUCTURED_PROMPT, return_tensors='pt')[0].tolist()

nw_s     = gen(STRUCTURED_PROMPT)
fixed_s  = gen(STRUCTURED_PROMPT, watermark_processor)
adp_s    = gen(STRUCTURED_PROMPT, adaptive_processor)
forced_s = generate_until_detected(model, prompt_ids_struct, watermark_processor, detector,
                                    max_tokens=400, device=DEVICE)

rows = [
    ('No watermark', nw_s),
    ('Fixed δ',      fixed_s),
    ('Adaptive δ',   adp_s),
    ('Forced',       forced_s),
]

print(f'{'Condition':<14}  {'Tokens':>6}  {'Z-score':>8}  {'Detected':>9}  Text preview')
print('-' * 80)
for label, toks in rows:
    r = detector.detect(toks)
    preview = tokenizer.decode(toks[:20], skip_special_tokens=True).replace('\n',' ')
    print(f'{label:<14}  {len(toks):>6}  {r["z_score"]:>8.2f}  {str(r["is_watermarked"]):>9}  {preview}')


### Bulk evaluation results

After running `generate.py` and `evaluate.py`, load the saved summary and plot the four conditions side by side.
```bash
!python generate.py --num_samples 500 --output_file data/generations.jsonl
!python evaluate.py --input_file data/generations.jsonl
```

In [ ]:
import json, os
from pathlib import Path

eval_path = Path('data/generations.eval.json')
if not eval_path.exists():
    print('Run generate.py + evaluate.py first, then re-run this cell.')
else:
    res = json.loads(eval_path.read_text())

    conditions = ['Fixed δ', 'Adaptive δ', 'Forced']
    tpr_vals   = [res['tpr'],       res.get('adp_tpr'),    res.get('forced_tpr')]
    auc_vals   = [res['auc'],       res.get('adp_auc'),    res.get('forced_auc')]
    z_means    = [res['w_z_mean'],  res.get('adp_z_mean'), res.get('forced_z_mean')]
    nw_tpr     = res['fpr']  # false-positive rate of baseline as reference

    # Filter to conditions that exist in the results
    present = [(c, t, a, z) for c,t,a,z in zip(conditions,tpr_vals,auc_vals,z_means) if t is not None]
    conditions, tpr_vals, auc_vals, z_means = zip(*present)

    x = range(len(conditions))
    colors = ['steelblue', 'darkorange', 'seagreen']

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))

    axes[0].bar(conditions, tpr_vals, color=colors[:len(conditions)])
    axes[0].axhline(nw_tpr, color='red', linestyle='--', linewidth=1, label=f'FPR baseline ({nw_tpr:.3f})')
    axes[0].set_ylim(0, 1.05)
    axes[0].set_ylabel('TPR')
    axes[0].set_title('Detection rate (TPR @ z>4.0)')
    axes[0].legend(fontsize=8)

    axes[1].bar(conditions, auc_vals, color=colors[:len(conditions)])
    axes[1].set_ylim(0.5, 1.02)
    axes[1].set_ylabel('AUC')
    axes[1].set_title('ROC AUC')

    axes[2].bar(conditions, z_means, color=colors[:len(conditions)])
    axes[2].axhline(res['nw_z_mean'], color='gray', linestyle='--', linewidth=1,
                    label=f'No-watermark mean ({res["nw_z_mean"]:.2f})')
    axes[2].set_ylabel('Mean z-score')
    axes[2].set_title('Mean z-score')
    axes[2].legend(fontsize=8)

    for ax in axes:
        ax.tick_params(axis='x', rotation=15)

    params = f'γ={res.get("gamma", 0.25)}  δ={res.get("delta", 2.0)}  T=200  n=500'
    fig.suptitle(f'Watermark detection — OPT-1.3B on C4 RealNewsLike\n{params}', fontsize=11)
    plt.tight_layout()
    plt.savefig('bulk_results.png', dpi=150, bbox_inches='tight')
    plt.show()

    if res.get('forced_len_mean'):
        print(f'Forced generation avg length: {res["forced_len_mean"]:.1f} tokens '
              f'(vs fixed {200} tokens, {res["forced_len_mean"]/200:.1f}x)')
